# B — Event-Driven FL: Independent Virtual Clients

**Faithful experimental settings from your original notebook**: 10 clients, MNIST, identical seed and Dirichlet/IID partitioning, SimpleNN (784→128→64→10), SGD + momentum, 2 local epochs, batch size 64, 5 global rounds, fixed virtual time quantum of 1 second, original speeds and network latencies, and wait-for-all weighted FedAvg. Each client keeps its weights, momentum, epoch, batch cursor, and compute credit across quanta. The original `SimulatedClient` implementation and its end-of-quantum training-completion convention are reused without modification.

**Important:** Neither version uses wall-clock training duration as simulated time. PyTorch still executes real forward/backward/optimizer steps sequentially on the Colab runtime. Both versions model virtual time; neither provides physically parallel GPU/CPU training. To compare fairly with the uploaded original, run all three notebooks under the same settings, seed, and Colab runtime. Results can differ if event order changes RNG consumption, particularly when data are shuffled globally. This implementation keeps the original shared Python RNG, so independent clients may consume shuffles in a different order.


## 1. Original experiment configuration

In [ ]:
import copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
NUM_CLIENTS = 10
NUM_GLOBAL_ROUNDS = 5
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
LEARNING_RATE = 0.03
MOMENTUM = 0.9
TRAIN_SAMPLES = 12_000
TEST_SAMPLES = 2_000
TIME_QUANTUM = 1.0
CLIENT_SPEEDS = [1.0, 1.5, 0.5, 1.0, 2.0, 0.35, 1.2, 0.75, 1.0, 0.6]
DOWNLOAD_LATENCY = [0, 0, 2, 0, 1, 5, 0, 3, 0, 4]
UPLOAD_LATENCY = [1, 1, 4, 1, 1, 6, 2, 3, 1, 5]
PARTITION_MODE = 'dirichlet'  # or 'iid'
DIRICHLET_ALPHA = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('Running on:', DEVICE)
# Concise console logging: queue preview only at the start of each round.
QUEUE_PREVIEW_SIZE = 6


## 2. Original MNIST selection and client partitioning

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset_full = datasets.MNIST('./data', train=False, download=True, transform=transform)
rng = np.random.default_rng(SEED)
train_indices = rng.choice(len(train_dataset_full), size=TRAIN_SAMPLES, replace=False)
test_indices = rng.choice(len(test_dataset_full), size=TEST_SAMPLES, replace=False)
test_dataset = Subset(test_dataset_full, test_indices.tolist())

def iid_partition(indices, num_clients):
    shuffled = np.array(indices).copy()
    np.random.default_rng(SEED).shuffle(shuffled)
    return [chunk.tolist() for chunk in np.array_split(shuffled, num_clients)]

def dirichlet_partition(dataset, selected_indices, num_clients, alpha):
    rng = np.random.default_rng(SEED)
    labels = np.asarray(dataset.targets)[selected_indices]
    # Retry if a client would have no samples.
    for _ in range(100):
        result = [[] for _ in range(num_clients)]
        for class_id in np.unique(labels):
            indices = np.asarray(selected_indices)[labels == class_id].copy()
            rng.shuffle(indices)
            p = rng.dirichlet(np.full(num_clients, alpha))
            chunks = np.split(indices, (np.cumsum(p)[:-1] * len(indices)).astype(int))
            for i, chunk in enumerate(chunks): result[i].extend(chunk.tolist())
        if all(result):
            for group in result: rng.shuffle(group)
            return result
    raise RuntimeError('Could not form nonempty client partitions; increase alpha.')

client_partitions = (iid_partition(train_indices, NUM_CLIENTS) if PARTITION_MODE == 'iid'
    else dirichlet_partition(train_dataset_full, train_indices, NUM_CLIENTS, DIRICHLET_ALPHA))
print('Samples per client:', [len(p) for p in client_partitions])

## 3. Original neural network, client training, FedAvg, evaluation

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

class SimulatedClient:
    def __init__(self, client_id, indices, speed, download_latency, upload_latency):
        self.client_id = client_id
        self.indices = list(indices)
        self.num_samples = len(indices)
        self.speed = speed
        self.download_latency = download_latency
        self.upload_latency = upload_latency

    def start_round(self, global_state, current_time, round_number):
        self.round_number = round_number
        self.model = SimpleNN().to(DEVICE)
        self.model.load_state_dict(copy.deepcopy(global_state))
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)
        self.epoch = 0
        self.batch_cursor = 0
        self.work_credit = 0.0
        self.training_finished = False
        self.update_arrived = False
        self.finish_time = None
        self.arrival_time = None
        self.ready_time = current_time + self.download_latency
        self.total_batches_trained = 0
        self.running_loss = 0.0
        self._shuffle_epoch()

    def _shuffle_epoch(self):
        self.shuffled_indices = self.indices.copy()
        random.shuffle(self.shuffled_indices)
        self.batch_cursor = 0

    def _train_one_batch(self):
        batch_ids = self.shuffled_indices[self.batch_cursor:self.batch_cursor + BATCH_SIZE]
        images, labels = zip(*(train_dataset_full[int(i)] for i in batch_ids))
        x = torch.stack(images).to(DEVICE)
        y = torch.tensor(labels, dtype=torch.long, device=DEVICE)
        self.model.train()
        self.optimizer.zero_grad()
        loss = F.cross_entropy(self.model(x), y)
        loss.backward()
        self.optimizer.step()
        self.running_loss += loss.item()
        self.total_batches_trained += 1
        self.batch_cursor += len(batch_ids)
        if self.batch_cursor >= len(self.shuffled_indices):
            self.epoch += 1
            if self.epoch >= LOCAL_EPOCHS: return True
            self._shuffle_epoch()
        return False

    def execute_time_slice(self, current_time, quantum):
        if self.training_finished or current_time < self.ready_time:
            return 0
        self.work_credit += self.speed * quantum
        batches = 0
        while self.work_credit + 1e-9 >= 1 and not self.training_finished:
            finished = self._train_one_batch()
            self.work_credit -= 1.0
            batches += 1
            if finished:
                self.training_finished = True
                # This baseline places completion at the END of the scheduled quantum.
                self.finish_time = current_time + quantum
                self.arrival_time = self.finish_time + self.upload_latency
        return batches

    def get_model_state(self):
        return {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}

def fedavg(states, weights):
    total = sum(weights)
    return {key: sum(states[i][key] * (weights[i] / total) for i in range(len(states)))
            for key in states[0]}

@torch.no_grad()
def evaluate(model):
    model.eval()
    loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        output = model(x)
        total_loss += F.cross_entropy(output, y, reduction='sum').item()
        correct += (output.argmax(1) == y).sum().item()
        total += y.numel()
    return {'loss': total_loss / total, 'accuracy': correct / total}

**Console logging:** one short client/version list and queue preview per round, followed by client results and the aggregation summary. Individual training quanta and full event histories are saved to CSV without overwhelming the output.


## 4. Discrete-event queue and reporting helpers

In [ ]:
import heapq
import itertools

# Events with identical timestamps use explicit priorities for deterministic behavior.
EVENT_PRIORITY = {'DOWNLOAD_COMPLETE': 0, 'UPDATE_ARRIVED': 1,
                  'QUANTUM_END': 2, 'DISPATCH': 3}

class EventQueue:
    def __init__(self):
        self.heap = []
        self.counter = itertools.count()

    def push(self, when, kind, client_id=-1):
        heapq.heappush(self.heap, (float(when), EVENT_PRIORITY[kind],
                                   next(self.counter), kind, client_id))

    def pop(self):
        when, _, _, kind, client_id = heapq.heappop(self.heap)
        return when, kind, client_id

    def __bool__(self):
        return bool(self.heap)


def log_event(round_number, when, client, event, batches=0):
    event_log.append({'round': round_number, 'virtual_time': when,
                      'client': client, 'event': event, 'batches': batches})


def summarize_round(round_number, round_start, round_end):
    global global_state
    for c in clients:
        client_round_metrics.append({'round': round_number, 'client': c.client_id,
            'speed': c.speed, 'samples': c.num_samples,
            'download_latency': c.download_latency, 'upload_latency': c.upload_latency,
            'finish_time': c.finish_time, 'arrival_time': c.arrival_time,
            'batches': c.total_batches_trained,
            'mean_train_loss': c.running_loss / max(1, c.total_batches_trained)})
    global_state = fedavg([c.get_model_state() for c in clients], [c.num_samples for c in clients])
    global_model.load_state_dict(global_state)
    stats = evaluate(global_model)
    duration = round_end - round_start
    round_metrics.append({'round': round_number, 'accuracy': stats['accuracy'],
                          'loss': stats['loss'], 'start_time': round_start,
                          'end_time': round_end, 'duration': duration})
    print(f'Round {round_number} | accuracy={stats["accuracy"]*100:.2f}% '
          f'| loss={stats["loss"]:.4f} | virtual duration={duration:.2f}s')


def queue_snapshot(queue, limit=QUEUE_PREVIEW_SIZE):
    """Display heap in actual priority order, NOT its internal heap-array order."""
    ordered = sorted(queue.heap)
    return [f"t={t:.1f} {kind} C{cid} (p={priority})"
            for t, priority, seq, kind, cid in ordered[:limit]]

def print_queue(queue, heading='NEXT EVENTS'):
    preview = queue_snapshot(queue)
    print(f"  {heading} | pending={len(queue.heap)} | ordered by (time, priority, insertion)")
    for pos, item in enumerate(preview, 1):
        print(f"    {pos:02d}. {item}")
    if len(queue.heap) > len(preview):
        print(f"    ... {len(queue.heap)-len(preview)} more events")


## 5. Event-driven round scheduler

Each client has an independent **virtual** training timeline, beginning after its own download completes. Each quantum-end event executes the client’s real PyTorch batches and schedules either the next quantum or the upload. These executions are still sequential in physical wall-clock time.

In [ ]:
clients = [SimulatedClient(i, client_partitions[i], CLIENT_SPEEDS[i],
                           DOWNLOAD_LATENCY[i], UPLOAD_LATENCY[i])
           for i in range(NUM_CLIENTS)]
global_model = SimpleNN().to(DEVICE)
global_state = copy.deepcopy(global_model.state_dict())
virtual_time = 0.0
event_log, round_metrics, client_round_metrics = [], [], []
queue_history = []

# Record the shared initial model so comparisons begin at virtual time zero.
initial_eval = evaluate(global_model)
round_metrics.append({'round': 0, 'accuracy': initial_eval['accuracy'],
                      'loss': initial_eval['loss'], 'start_time': 0.0,
                      'end_time': 0.0, 'duration': 0.0})

for round_number in range(1, NUM_GLOBAL_ROUNDS + 1):
    round_start_time = virtual_time
    global_version = round_number - 1
    for c in clients:
        c.start_round(global_state, round_start_time, round_number)
        c.model_version = global_version  # explicit version carried by each client
    queue = EventQueue()
    for c in clients:
        queue.push(c.ready_time, 'DOWNLOAD_COMPLETE', c.client_id)
    arrived_count = 0
    processed_events = 0
    print(f"\n{'='*72}\nROUND {round_number}/{NUM_GLOBAL_ROUNDS} | global model v{global_version} | t={round_start_time:.1f}\n{'='*72}")
    print('CLIENTS STARTING (ID | model | speed | ready time)')
    for c in clients:
        print(f'  C{c.client_id:02d} | v{c.model_version} | {c.speed:.2f} batches/s | t={c.ready_time:.1f}')
    print_queue(queue, 'NEXT IN PRIORITY QUEUE')

    while queue:
        when, kind, client_id = queue.pop()
        processed_events += 1
        virtual_time = when
        c = clients[client_id]
        # Queue snapshots are retained for optional CSV inspection, not printed per event.
        if processed_events == 1:
            queue_history.append({'round': round_number, 'time': when, 'processed': processed_events,
                                  'pending': len(queue.heap), 'next_events': ' | '.join(queue_snapshot(queue))})
        if kind == 'DOWNLOAD_COMPLETE':
            log_event(round_number, when, client_id, 'DOWNLOAD_COMPLETE')
            log_event(round_number, when, client_id, 'QUANTUM_START')
            queue.push(when + TIME_QUANTUM, 'QUANTUM_END', client_id)

        elif kind == 'QUANTUM_END':
            # Physical PyTorch training happens now; virtual times are simulated.
            batches = c.execute_time_slice(when - TIME_QUANTUM, TIME_QUANTUM)
            log_event(round_number, when, client_id, 'QUANTUM_END', batches)
            if c.training_finished:
                log_event(round_number, when, client_id, 'TRAINING_COMPLETE')
                queue.push(c.arrival_time, 'UPDATE_ARRIVED', client_id)
            else:
                log_event(round_number, when, client_id, 'QUANTUM_START')
                queue.push(when + TIME_QUANTUM, 'QUANTUM_END', client_id)

        elif kind == 'UPDATE_ARRIVED':
            if not c.update_arrived:
                c.update_arrived = True
                arrived_count += 1
                log_event(round_number, when, client_id, 'UPDATE_ARRIVED')
                if arrived_count == NUM_CLIENTS:
                    break

    assert arrived_count == NUM_CLIENTS, 'Round ended before receiving all clients.'
    print('CLIENT RESULTS (ID | model | batches | finish | arrived)')
    for c in clients:
        print(f'  C{c.client_id:02d} | v{c.model_version} | {c.total_batches_trained:3d} | '
              f't={c.finish_time:.1f} | t={c.arrival_time:.1f}')
    print(f'AGGREGATE | {arrived_count}/{NUM_CLIENTS} updates | '
          f'FedAvg v{global_version} -> v{global_version+1} at t={virtual_time:.1f}')
    summarize_round(round_number, round_start_time, virtual_time)
    virtual_time += TIME_QUANTUM  # same inter-round virtual gap as original
print('Independent-client DES simulation complete.')


## 6. Metrics, event log, plots and CSV exports

Compare the resulting CSVs against the original notebook. Round lengths can differ because independent virtual clients have overlapping simulated compute. These are synchronous wait-for-all FedAvg baselines, not FedAsync. The original training-completion-at-quantum-end approximation is intentionally preserved.

In [ ]:
events_df = pd.DataFrame(event_log)
metrics_df = pd.DataFrame(round_metrics)
timing_df = pd.DataFrame(client_round_metrics)
print('GLOBAL METRICS')
display(metrics_df)
# Detailed traces are saved to CSV, rather than flooding notebook output.

plt.figure(figsize=(8, 4))
plt.plot(metrics_df['round'], metrics_df['accuracy'] * 100, marker='o')
plt.xlabel('Global round'); plt.ylabel('Test accuracy (%)')
plt.title('Global model convergence'); plt.grid(True); plt.show()

last_round = timing_df[timing_df['round'] == NUM_GLOBAL_ROUNDS]
plt.figure(figsize=(9, 4))
plt.bar(last_round['client'].astype(str), last_round['arrival_time'] - metrics_df.iloc[-1]['start_time'])
plt.xlabel('Client ID'); plt.ylabel('Virtual time since round start')
plt.title('Final-round update arrivals'); plt.show()

metrics_df.to_csv('round_metrics.csv', index=False)
timing_df.to_csv('client_timing.csv', index=False)
events_df.to_csv('scheduler_events.csv', index=False)
print('Saved round_metrics.csv, client_timing.csv, scheduler_events.csv')

queue_df = pd.DataFrame(queue_history)
queue_df.to_csv('priority_queue_snapshots.csv', index=False)

# Visual timeline: arrivals and completion times for each client in the final round.
plt.figure(figsize=(11, 5))
for _, r in last_round.iterrows():
    cid = int(r['client'])
    plt.plot([r['finish_time'], r['arrival_time']], [cid, cid], marker='o', linewidth=2)
plt.yticks(range(NUM_CLIENTS), [f'C{i:02d} (v{NUM_GLOBAL_ROUNDS-1})' for i in range(NUM_CLIENTS)])
plt.xlabel('Virtual time'); plt.ylabel('Client / model version')
plt.title('Final round: training finished → update arrived (upload latency)')
plt.grid(axis='x', alpha=.3); plt.tight_layout(); plt.show()
print('Also saved priority_queue_snapshots.csv (detailed records available in CSV files).')
